# Bab 08 · NumPy: Array, Broadcasting, dan Vektorisasi

**Notebook praktikum mahasiswa**  
Versi 2.0 · Pemrograman Komputer

- Membaca bentuk array dan makna axis.
- Mencegah perubahan input melalui view.
- Melakukan operasi vektor dan standardisasi kolom.

### Petunjuk menjalankan sel · versi 2.0

Jalankan sel berurutan dari atas ke bawah. Setiap fungsi mandiri diletakkan pada sel tersendiri; sel pemanggilan atau pengujiannya menyusul setelah definisi. Setelah menyunting fungsi, jalankan ulang sel definisinya, lalu sel pengujiannya.

Sel persiapan dan fungsi pemeriksa cukup dijalankan; bagian yang Anda kerjakan ditandai **[ISI KODE]**. Metode yang membentuk satu kelas serta fungsi bersarang tetap disatukan karena merupakan satu kesatuan Python.

## Alur praktikum

**Duga → Jalankan → Selidiki → Isi kode → Periksa → Jelaskan**

Perkiraan waktu: 90–120 menit. Kerjakan berpasangan; tukar peran penulis kode dan pemeriksa setiap dua latihan.

| Penanda | Yang Anda kerjakan |
|---|---|
| [BACA] | Pahami konsep, kontrak fungsi, dan kasus batas. |
| [DUGA] | Tulis prediksi sebelum menjalankan contoh. |
| [COBA] | Jalankan contoh dan ubah satu hal untuk menyelidiki hasilnya. |
| [ISI KODE] | Lengkapi fungsi atau kelas; pertahankan nama dan parameternya. |
| [CEK OTOMATIS] | Jalankan pengujian yang terlihat, lalu gunakan pesannya untuk memperbaiki kode. |
| [REFLEKSI] | Jelaskan alasan dan bukti, bukan hanya menyalin keluaran. |

Impor file `.ipynb` ini ke notebook Python di Kaggle. Gunakan CPU; data kecil disediakan dalam notebook. Jalankan sel dari atas ke bawah. Pustaka yang diperlukan diimpor pada sel persiapan; tidak ada perintah instalasi atau unduhan.

`BELUM DIISI` adalah status normal pada notebook awal. Ganti `raise BelumDiisi()` dengan pekerjaan Anda. `LULUS` berarti memenuhi kasus uji yang tersedia, bukan bukti bahwa semua kemungkinan input sudah benar. Sel pengujian harus tetap utuh.

Jika kode berulang tanpa selesai, hentikan eksekusi, periksa batas perulangan, lalu jalankan ulang. Sebelum mengumpulkan, mulai ulang sesi Python dan jalankan seluruh sel agar hasil tidak bergantung pada variabel lama.

### Identitas

- Nama: …
- NIM: …
- Rekan diskusi: …
- Tanggal: …

**Persiapan dan pengaturan** · bagian 1 dari 9

In [ ]:
# [COBA] Jalankan sekali di awal; pemeriksaan tersedia untuk dibaca.
import math
import sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

**Definisi `BelumDiisi`** · bagian 2 dari 9

In [ ]:
class BelumDiisi(Exception):
    """Penanda latihan yang belum dikerjakan."""

**Definisi `sama`** · bagian 3 dari 9

In [ ]:
def sama(aktual, harapan):
    assert aktual == harapan, f"Diharapkan {harapan!r}; diperoleh {aktual!r}"

**Definisi `dekat`** · bagian 4 dari 9

In [ ]:
def dekat(aktual, harapan, atol=1e-8, rtol=1e-7):
    assert math.isclose(
        aktual, harapan, abs_tol=atol, rel_tol=rtol
    ), f"Diharapkan sekitar {harapan!r}; diperoleh {aktual!r}"

**Definisi `harus_galat`** · bagian 5 dari 9

In [ ]:
def harus_galat(jenis, panggil):
    try:
        panggil()
    except BelumDiisi:
        raise
    except jenis:
        return
    raise AssertionError(f"Seharusnya memunculkan {jenis.__name__}")

**Persiapan dan pengaturan** · bagian 6 dari 9

In [ ]:
DAFTAR_UJI = {}

**Definisi `cek`** · bagian 7 dari 9

In [ ]:
def cek(nomor, fungsi_uji, tampil=True):
    DAFTAR_UJI[nomor] = fungsi_uji
    try:
        fungsi_uji()
        status, pesan = "LULUS", "Semua kasus uji pada latihan ini sesuai."
    except BelumDiisi:
        status, pesan = (
            "BELUM DIISI",
            "Lengkapi sel [ISI KODE], jalankan, lalu ulangi pemeriksaan.",
        )
    except AssertionError as err:
        status, pesan = (
            "PERLU PERBAIKAN",
            str(err) or "Hasil belum sesuai kontrak latihan.",
        )
    except Exception as err:
        status, pesan = "GALAT", f"{type(err).__name__}: {err}"
    if tampil:
        print(f"Latihan {nomor} | {status}\n{pesan}")
    return status

**Definisi `rekap`** · bagian 8 dari 9

In [ ]:
def rekap():
    # Uji ulang fungsi terkini agar rekap tidak memakai status lama.
    hasil = {
        nomor: cek(nomor, uji, tampil=False)
        for nomor, uji in sorted(DAFTAR_UJI.items())
    }
    for nomor, status in hasil.items():
        print(f"  Latihan {nomor}: {status}")
    lulus = sum(s == "LULUS" for s in hasil.values())
    print(f"\nKemajuan uji otomatis: {lulus}/{JUMLAH_LATIHAN} latihan lulus.")
    print(
        "Refleksi, penjelasan, dan kualitas penyajian diperiksa bersama asisten."
    )
    return hasil

**Persiapan dan pengaturan** · bagian 9 dari 9

In [ ]:
print("Python:", sys.version.split()[0])
print("Siap. Jalankan notebook dari atas ke bawah.")
JUMLAH_LATIHAN = 4
import numpy as np

print("NumPy:", np.__version__)

## [BACA] Konsep inti

Array menyimpan elemen bertipe seragam. Pada array 2D, agregasi `axis=0` meringkas baris sehingga tersisa satu nilai per kolom. Pengirisan dasar biasanya menghasilkan view yang berbagi memori. Broadcasting mencocokkan dimensi dari kanan; dimensi cocok jika sama atau salah satunya 1. Standardisasi harus menggunakan statistik data latih saat diterapkan ke data baru.

## [DUGA] Prediksi sebelum eksekusi

Apa isi a setelah b diubah, dan apa bentuk u + v?

**Prediksi saya:** …

**Alasan:** …

In [ ]:
# [COBA]
a = np.arange(6)
b = a[1:4]
b[0] = 99
print("asal:", a)
u = np.array([[1], [2], [3]])
v = np.array([10, 20])
print("bentuk:", (u + v).shape)
print("jumlah kolom:", (u + v).sum(axis=0))

**[REFLEKSI]** Apa perbedaan prediksi dan hasil? Ubah satu input pada contoh, tulis hasilnya, lalu jelaskan konsep yang ditunjukkan.

**Jawaban:** …

## Latihan 1 · Transformasi vektor

**[ISI KODE]**

Buat `ubah_suhu(celsius)` yang menerima array 1D float dan mengembalikan array Fahrenheit dengan rumus F = 1,8 C + 32. Input tetap utuh, termasuk input kosong. Gunakan operasi array.

> Petunjuk: Operasi aritmetika dengan skalar berlaku pada seluruh elemen.

In [ ]:
# [ISI KODE]
def ubah_suhu(celsius):
    raise BelumDiisi()

**Definisi `uji_01`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_01():
    a = np.array([-40.0, 0.0, 100.0])
    awal = a.copy()
    hasil = ubah_suhu(a)
    assert isinstance(hasil, np.ndarray), "Kembalikan ndarray."
    np.testing.assert_allclose(hasil, [-40, 32, 212])
    np.testing.assert_array_equal(a, awal)
    sama(ubah_suhu(np.array([])).shape, (0,))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(1, uji_01)

## Latihan 2 · Salinan irisan yang aman

**[ISI KODE]**

Buat `potong_mandiri(a, awal, akhir)` yang mengembalikan salinan a[awal:akhir] untuk array 1D. Mengubah hasil tidak boleh mengubah a. Batas mengikuti aturan slicing Python.

> Petunjuk: Pengirisan dan penyalinan adalah dua langkah yang berbeda.

In [ ]:
# [ISI KODE]
def potong_mandiri(a, awal, akhir):
    raise BelumDiisi()

**Definisi `uji_02`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_02():
    a = np.arange(6)
    b = potong_mandiri(a, 1, 4)
    np.testing.assert_array_equal(b, [1, 2, 3])
    assert not np.shares_memory(a, b), "Hasil tidak boleh berbagi memori."
    b[0] = 88
    np.testing.assert_array_equal(a, np.arange(6))
    sama(potong_mandiri(a, 3, 3).shape, (0,))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(2, uji_02)

## Latihan 3 · Selisih pasangan dengan broadcasting

**[ISI KODE]**

Buat `selisih_pasangan(a, b)` untuk dua array 1D. Kembalikan matriks D dengan D[i,j] = a[i] − b[j], bentuk `(len(a), len(b))`. Gunakan broadcasting; dukung array kosong.

> Petunjuk: Tambahkan satu sumbu menggunakan `None` atau `reshape`.

In [ ]:
# [ISI KODE]
def selisih_pasangan(a, b):
    raise BelumDiisi()

**Definisi `uji_03`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_03():
    np.testing.assert_array_equal(
        selisih_pasangan(np.array([1, 4]), np.array([10, 20, 30])),
        [[-9, -19, -29], [-6, -16, -26]],
    )
    sama(selisih_pasangan(np.array([]), np.array([1, 2])).shape, (0, 2))
    sama(selisih_pasangan(np.array([1]), np.array([])).shape, (1, 0))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(3, uji_03)

## Latihan 4 · Standardisasi dan kolom konstan

**[ISI KODE]**

Buat `standardisasi(D)` untuk array float 2D berhingga dan tidak kosong. Kembalikan `(Z, mu, sd)` dengan rerata per kolom dan simpangan baku populasi (`ddof=0`). Tolak kolom sd = 0 dengan `ValueError` yang menyebut indeks kolom. D tidak boleh berubah.

> Petunjuk: Hitung mu dan sd sepanjang axis=0; simpan keduanya untuk data baru.

In [ ]:
# [ISI KODE]
def standardisasi(D):
    raise BelumDiisi()

**Definisi `uji_04`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_04():
    D = np.array([[1.0, 10.0], [3.0, 20.0], [5.0, 30.0]])
    awal = D.copy()
    Z, mu, sd = standardisasi(D)
    np.testing.assert_allclose(mu, [3, 20])
    np.testing.assert_allclose(sd, [np.sqrt(8 / 3), np.sqrt(200 / 3)])
    np.testing.assert_allclose(Z.mean(axis=0), [0, 0], atol=1e-12)
    np.testing.assert_allclose(Z.std(axis=0), [1, 1])
    np.testing.assert_allclose(Z * sd + mu, D)
    np.testing.assert_array_equal(D, awal)
    try:
        standardisasi(np.array([[1.0, 2.0], [3.0, 2.0]]))
    except ValueError as err:
        assert "1" in str(err), "Sebut indeks kolom konstan: 1."
    else:
        raise AssertionError("Kolom konstan harus ditolak.")

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(4, uji_04)

## [REFLEKSI] Refleksi akhir

Bagian ini membantu Anda merangkum pemahaman, mengenali kesulitan, dan menjelaskan alasan di balik kode. Tulis jawaban singkat berdasarkan percobaan Anda; bukan sekadar menyalin keluaran.

1. Pilih satu latihan. Jelaskan alur kode Anda dengan satu contoh input dan hasilnya.
2. Tuliskan satu kesalahan yang sempat terjadi, penyebabnya, dan cara memperbaikinya.
3. Usulkan satu kasus uji tambahan yang belum tercakup. Nyatakan hasil yang Anda harapkan dan alasannya.
4. Apa batas kesimpulan yang boleh dibuat dari hasil praktikum ini?

**Jawaban:** …

### Tantangan pengembangan

Tambahkan kasus uji usulan Anda pada sel di bawah. Pastikan kasus tersebut bisa membedakan implementasi benar dan satu kesalahan yang masuk akal. Diskusikan dengan asisten sebelum mengubah kontrak fungsi.

In [ ]:
# [ISI KODE OPSIONAL] Tambahkan eksperimen atau pengujian buatan Anda.
# Jelaskan harapan Anda pada komentar sebelum menjalankannya.

In [ ]:
# [CEK OTOMATIS] Uji ulang seluruh latihan yang sudah didaftarkan.
status_akhir = rekap()

## Sebelum mengumpulkan

- [ ] Identitas dan prediksi sudah diisi.
- [ ] Semua latihan sudah dikerjakan dan diperiksa dari sesi baru.
- [ ] refleksi akhir berisi penjelasan dengan bukti keluaran.
- [ ] Notebook disimpan dengan nama dan NIM; jangan hanya mengumpulkan HTML.

Rubrik diskusi: ketepatan kode 60%, penjelasan dan kasus batas 25%, keterbacaan serta kemampuan dijalankan ulang 15%. Rekap otomatis membantu belajar; penilaian akhir tetap memerlukan pemeriksaan asisten.

Rujukan: bab yang bersesuaian pada buku *Python untuk Machine Learning dan Data Science* dan modul praktikum. Latihan di notebook ini merupakan adaptasi terarah untuk praktikum, bukan seluruh soal akhir bab.